In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
Project_DIR = Path.cwd().resolve()
if Project_DIR.name == "notebook":
    Project_DIR = Project_DIR.parent

Data_DIR = Project_DIR / "DATA"
df = pd.read_csv(Data_DIR / "dataset.csv", encoding="latin-1")
df.head()

,text_type,text
0,spam,naturally irresistible your corporate identity...
1,spam,the stock trading gunslinger fanny is merrill ...
2,spam,unbelievable new homes made easy im wanting to...
3,spam,4 color printing special request additional in...
4,spam,do not have money get software cds from here s...


**Data Preprocessing**

In [8]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Convert text to lowercase
    text = text.lower()

    # Preserve useful spam indicators as tokens
    text = re.sub(r"http\S+|www\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)
    text = re.sub(r"\d+", " number ", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["cleaned_text"] = df["text"].apply(clean_text)
df["cleaned_text"].head()

0    naturally irresistible your corporate identity...
1    the stock trading gunslinger fanny is merrill ...
2    unbelievable new homes made easy im wanting to...
3    number color printing special request addition...
4    do not have money get software cds from here s...
Name: cleaned_text, dtype: object

Tokenenization(auto)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=False,
    min_df=2,        # Ignore words appearing in fewer than 2 messages
    max_df=0.95,     # Ignore words appearing in over 95% of messages
    max_features=5000
)

X = vectorizer.fit_transform(df["cleaned_text"])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 650243 stored elements and shape (20348, 5000)>

In [14]:
feature_names = vectorizer.get_feature_names_out()

X_table = pd.DataFrame.sparse.from_spmatrix(
    X,
    columns=feature_names
)

X_table.head()

,aa,aac,aap,ab,abilities,ability,able,about,above,absence,...,zee,zero,zhang,zhendong,zimbabwe,zimin,ziminforwarded,zip,zone,zzzz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
X.shape
X_table.shape
X_table.columns[:20]

Index(['aa', 'aac', 'aap', 'ab', 'abilities', 'ability', 'able', 'about',
       'above', 'absence', 'absolutely', 'abstract', 'abt', 'ac', 'academic',
       'acc', 'accept', 'acceptable', 'acceptance', 'accepted'],
      dtype='object')

In [16]:
from scipy.sparse import save_npz
import joblib

save_npz(Data_DIR / "tfidf_features.npz", X)
joblib.dump(vectorizer, Data_DIR / "tfidf_vectorizer.pkl")

df[["text_type"]].to_csv(
    Data_DIR / "labels.csv",
    index=False
)